In [ ]:

pip install rasterio numpy pandas matplotlib pillow


In [ ]:
import shutil
print(shutil.which("ffmpeg"))

/usr/bin/ffmpeg


RasterioIOError: C:\GIS\Day09_Expanding_City/Tashkent_First_Built_Year.tif: No such file or directory

In [ ]:

from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:

import os

folder = '/content/drive/MyDrive/Day09_Expanding_City'

print("Files in project folder:")

for file in os.listdir(folder):
    print(file)


Files in project folder:
Untitled0.ipynb
Tashkent_Urban_Expansion_1975_2025.mp4
Tashkent_First_Built_Year.tif


RasterioIOError: C:\GIS\Day09_Expanding_City/Tashkent_First_Built_Year.tif: No such file or directory

In [9]:

!pip -q install rasterio pandas numpy matplotlib

!apt-get -qq update
!apt-get -qq install -y ffmpeg

from google.colab import drive
drive.mount('/content/drive')

import os
import shutil

print("FFmpeg:", shutil.which("ffmpeg"))
print("Google Drive connected!")


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
FFmpeg: /usr/bin/ffmpeg
Google Drive connected!


Both input files found!

RASTER INFORMATION
------------------
CRS: EPSG:32642
Resolution: (100.0, 100.0)
Dimensions: 798 800
Extent: BoundingBox(left=480300.0, bottom=4532000.0, right=560100.0, top=4612000.0)

HISTORICAL STATISTICS
---------------------
    year  system:index  built_surface_km2  footprint_km2  \
0   1975             0         147.785582     432.905742   
1   1980             1         165.353835     494.842970   
2   1985             2         188.782417     592.149132   
3   1990             3         216.462065     689.931990   
4   1995             4         249.856097     794.143339   
5   2000             5         287.533237     861.069146   
6   2005             6         312.127087     948.240698   
7   2010             7         339.998335    1009.526305   
8   2015             8         367.157195    1054.970792   
9   2020             9         381.655776    1096.260354   
10  2025            10         390.475748    1133.159763   

                        

Input files successfully located!

RASTER INFORMATION
CRS: EPSG:32642
Resolution: (100.0, 100.0)
Dimensions: 798 800

AUTOMATIC MAP ZOOM
Urban extent calculated successfully.

HISTORICAL STATISTICS
 year  system:index  built_surface_km2  footprint_km2                                   .geo  growth_percent  net_increase_km2
 1975             0         147.785582     432.905742 {"type":"MultiPoint","coordinates":[]}        0.000000          0.000000
 1980             1         165.353835     494.842970 {"type":"MultiPoint","coordinates":[]}       11.887663         17.568253
 1985             2         188.782417     592.149132 {"type":"MultiPoint","coordinates":[]}       27.740754         40.996835
 1990             3         216.462065     689.931990 {"type":"MultiPoint","coordinates":[]}       46.470354         68.676483
 1995             4         249.856097     794.143339 {"type":"MultiPoint","coordinates":[]}       69.066626        102.070515
 2000             5         287.533237  

Both input files found!

RASTER INFORMATION
------------------
CRS: EPSG:32642
Resolution: (100.0, 100.0)
Dimensions: 798 800
Extent: BoundingBox(left=480300.0, bottom=4532000.0, right=560100.0, top=4612000.0)

HISTORICAL STATISTICS
---------------------
    year  system:index  built_surface_km2  footprint_km2  \
0   1975             0         147.785582     432.905742   
1   1980             1         165.353835     494.842970   
2   1985             2         188.782417     592.149132   
3   1990             3         216.462065     689.931990   
4   1995             4         249.856097     794.143339   
5   2000             5         287.533237     861.069146   
6   2005             6         312.127087     948.240698   
7   2010             7         339.998335    1009.526305   
8   2015             8         367.157195    1054.970792   
9   2020             9         381.655776    1096.260354   
10  2025            10         390.475748    1133.159763   

                        

In [28]:

# ==============================================================
# DAY 09 — THE EXPANDING CITY: TASHKENT 1975–2025
# ==============================================================
# MAP-DOMINANT GEOSPATIAL ANIMATION
#
# 1. Large historical urban development map
# 2. Synchronized urban growth graph
# 3. Dynamic year counter
# 4. Built-up surface statistics
# 5. Chronological colour gradient legend
#
# DATA: GHSL P2023A
# SOFTWARE: Python / Google Colab
# OUTPUT: MP4
# ==============================================================


# ==============================================================
# 1. IMPORT LIBRARIES
# ==============================================================

import os
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
import matplotlib.animation as animation

from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import Video, display


# ==============================================================
# 2. FILE PATHS
# ==============================================================

folder = '/content/drive/MyDrive/Day09_Expanding_City'

raster_path = os.path.join(
    folder,
    'Tashkent_First_Built_Year.tif'
)

csv_path = os.path.join(
    folder,
    'Tashkent_Urban_Growth_Statistics.csv'
)

output_path = os.path.join(
    folder,
    'Tashkent_Urban_Expansion_1975_2025.mp4'
)

# Check input files

if not os.path.exists(raster_path):
    raise FileNotFoundError(f'Raster not found: {raster_path}')

if not os.path.exists(csv_path):
    raise FileNotFoundError(f'CSV not found: {csv_path}')

print('Both input files found!')


# ==============================================================
# 3. HISTORICAL YEARS AND COLOUR PALETTE
# ==============================================================

years = [
    1975, 1980, 1985, 1990, 1995,
    2000, 2005, 2010, 2015, 2020, 2025
]

colors = [
    '#180B35',  # Present by 1975
    '#32105B',  # 1975–1980
    '#54166D',  # 1980–1985
    '#79196F',  # 1985–1990
    '#A52269',  # 1990–1995
    '#C93454',  # 1995–2000
    '#E34A3A',  # 2000–2005
    '#F36B21',  # 2005–2010
    '#F99D0B',  # 2010–2015
    '#FFC52F',  # 2015–2020
    '#FFF19A'   # 2020–2025
]

cmap = ListedColormap(colors)

boundaries = np.arange(1972.5, 2028, 5)

norm = BoundaryNorm(
    boundaries,
    cmap.N
)

background = '#EEE9DF'
dark = '#332333'
accent = '#B94E48'


# ==============================================================
# 4. LOAD FIRST BUILT-UP YEAR RASTER
# ==============================================================

with rasterio.open(raster_path) as src:

    raster = src.read(1, masked=True)

    raster_data = raster.astype(
        'float32'
    ).filled(np.nan)

    raster_bounds = src.bounds

    raster_extent = [
        raster_bounds.left,
        raster_bounds.right,
        raster_bounds.bottom,
        raster_bounds.top
    ]

    raster_transform = src.transform
    raster_crs = src.crs

    print('\nRASTER INFORMATION')
    print('CRS:', raster_crs)
    print('Resolution:', src.res)
    print('Dimensions:', src.width, src.height)
    print('Extent:', raster_bounds)


# ==============================================================
# 5. AUTOMATIC URBAN DEVELOPMENT EXTENT
# ==============================================================

# Identify all valid built-up pixels

valid_rows, valid_cols = np.where(
    np.isfinite(raster_data)
    & (raster_data > 0)
)

if len(valid_rows) == 0:
    raise ValueError('No valid built-up pixels found!')

# Find geographical boundaries of built-up pixels
# Assumes a north-up raster.

xmin = (
    raster_transform.c
    + valid_cols.min() * raster_transform.a
)

xmax = (
    raster_transform.c
    + (valid_cols.max() + 1) * raster_transform.a
)

y1 = (
    raster_transform.f
    + valid_rows.min() * raster_transform.e
)

y2 = (
    raster_transform.f
    + (valid_rows.max() + 1) * raster_transform.e
)

ymin = min(y1, y2)
ymax = max(y1, y2)

# Add 8% geographical padding

padding_x = (xmax - xmin) * 0.08
padding_y = (ymax - ymin) * 0.08

zoom_extent = [
    xmin - padding_x,
    xmax + padding_x,
    ymin - padding_y,
    ymax + padding_y
]

print('\nUrban development extent calculated!')


# ==============================================================
# 6. LOAD STATISTICS CSV
# ==============================================================

df = pd.read_csv(csv_path)

df['year'] = pd.to_numeric(df['year'])

df['built_surface_km2'] = pd.to_numeric(
    df['built_surface_km2']
)

df = df.sort_values('year').reset_index(drop=True)

df = df[
    df['year'].isin(years)
].copy()

# Check missing years

missing_years = [
    year for year in years
    if year not in df['year'].values
]

if missing_years:
    raise ValueError(f'Missing years: {missing_years}')

# Arrange chronologically

df = (
    df.set_index('year')
      .loc[years]
      .reset_index()
)

# Calculate growth relative to 1975

baseline = df.loc[
    df['year'] == 1975,
    'built_surface_km2'
].iloc[0]

if baseline <= 0:
    raise ValueError('Invalid 1975 baseline.')

df['growth_percent'] = (
    (df['built_surface_km2'] - baseline)
    / baseline
) * 100

df['net_increase_km2'] = (
    df['built_surface_km2'] - baseline
)

print('\nHISTORICAL STATISTICS')
print(df.to_string(index=False))


# ==============================================================
# 7. PREPARE GRAPH DATA
# ==============================================================

all_years = df['year'].to_numpy()

all_areas = df[
    'built_surface_km2'
].to_numpy()

max_area = all_areas.max()


# ==============================================================
# 8. FIGURE LAYOUT — MAP DOMINANT
# ==============================================================

fig = plt.figure(
    figsize=(16, 10),
    facecolor=background
)

# MANUAL AXIS POSITIONING
#
# Map: approximately 45% of canvas width
# Graph: approximately 30% of canvas width
#
# The map has more visual space than the previous version.

ax_map = fig.add_axes([
    0.025,  # Left
    0.155,  # Bottom
    0.55,   # Width
    0.70    # Height
])

ax_chart = fig.add_axes([
    0.665,
    0.235,
    0.30,
    0.48
])


# Main title

fig.suptitle(
    'URBAN EXPANSION',
    fontsize=27,
    fontweight='bold',
    color=dark,
    y=0.965
)



# ==============================================================
# 9. LARGE HISTORICAL MAP
# ==============================================================

ax_map.set_facecolor(background)

# Initial map: built-up area present by 1975

initial_map = np.ma.masked_where(
    (~np.isfinite(raster_data))
    | (raster_data <= 0)
    | (raster_data > 1975),
    raster_data
)

map_image = ax_map.imshow(
    initial_map,
    cmap=cmap,
    norm=norm,
    extent=raster_extent,
    origin='upper',
    interpolation='nearest'
)

# Zoom to urban development

ax_map.set_xlim(
    zoom_extent[0],
    zoom_extent[1]
)

ax_map.set_ylim(
    zoom_extent[2],
    zoom_extent[3]
)

# Preserve geographic proportions

ax_map.set_aspect(
    'equal',
    adjustable='box'
)

ax_map.set_title(
    'TASHKENT, UZBEKISTAN  |  1975–2025',
    fontsize=14,
    fontweight='bold',
    color=accent,
    pad=19
)

ax_map.set_xticks([])
ax_map.set_yticks([])

for spine in ax_map.spines.values():
    spine.set_color('#B9AA9A')
    spine.set_linewidth(1.3)


# Dynamic year label

year_label = ax_map.text(
    0.035,
    0.93,
    '1975',
    transform=ax_map.transAxes,
    fontsize=30,
    fontweight='bold',
    color='white',
    bbox=dict(
        facecolor=dark,
        edgecolor='none',
        alpha=0.95,
        pad=11
    )
)


# ==============================================================
# 10. STATISTICAL GRAPH
# ==============================================================

ax_chart.set_facecolor(background)

# Background historical trend

ax_chart.plot(
    all_years,
    all_areas,
    color='#D2C7BB',
    linewidth=2,
    marker='o',
    markersize=4,
    zorder=1
)

# Animated line

progress_line, = ax_chart.plot(
    [],
    [],
    color=accent,
    linewidth=2.8,
    marker='o',
    markersize=4,
    zorder=2
)

# Animated indicator

current_point, = ax_chart.plot(
    [],
    [],
    marker='o',
    markersize=10,
    color='#F36B21',
    markeredgecolor=dark,
    markeredgewidth=1.3,
    zorder=3
)

ax_chart.set_title(
    'URBAN GROWTH OVER TIME',
    fontsize=14,
    fontweight='bold',
    color=dark,
    pad=14
)

ax_chart.set_xlabel(
    'YEAR',
    fontsize=10,
    color=dark
)

ax_chart.set_ylabel(
    'BUILT-UP SURFACE (km²)',
    fontsize=10,
    color=dark
)

ax_chart.set_xlim(1972, 2028)

ax_chart.set_ylim(
    0,
    max_area * 1.2
)

ax_chart.set_xticks(years)

ax_chart.tick_params(
    axis='x',
    rotation=40,
    labelsize=8,
    colors=dark
)

ax_chart.tick_params(
    axis='y',
    labelsize=9,
    colors=dark
)

ax_chart.grid(
    True,
    linestyle='--',
    linewidth=0.6,
    alpha=0.3
)

for spine in ax_chart.spines.values():
    spine.set_color('#B9AA9A')


# ==============================================================
# 11. DYNAMIC STATISTICS ABOVE GRAPH
# ==============================================================

area_text = fig.text(
    0.815,
    0.855,
    '',
    ha='center',
    fontsize=15,
    fontweight='bold',
    color=accent
)

growth_text = fig.text(
    0.815,
    0.815,
    '',
    ha='center',
    fontsize=11,
    color=dark
)


# ==============================================================
# 12. TIMELINE LEGEND
# ==============================================================

legend_labels = [
    'By 1975',
    '1975–80',
    '1980–85',
    '1985–90',
    '1990–95',
    '1995–00',
    '2000–05',
    '2005–10',
    '2010–15',
    '2015–20',
    '2020–25'
]

legend_elements = [
    Patch(
        facecolor=colors[i],
        label=legend_labels[i]
    )
    for i in range(len(colors))
]

fig.legend(
    handles=legend_elements,
    loc='lower center',
    ncol=6,
    frameon=False,
    fontsize=9,
    bbox_to_anchor=(0.5, 0.045),
    title='BUILT-UP AREA',
    title_fontsize=10
)


# ==============================================================
# 13. ANIMATION FUNCTION
# ==============================================================

def update(frame):

    year = years[frame]

    # ----------------------------------------------------------
    # A. UPDATE CUMULATIVE MAP
    # ----------------------------------------------------------

    current_map = np.ma.masked_where(
        (~np.isfinite(raster_data))
        | (raster_data <= 0)
        | (raster_data > year),
        raster_data
    )

    map_image.set_data(current_map)

    # ----------------------------------------------------------
    # B. UPDATE YEAR LABEL
    # ----------------------------------------------------------

    year_label.set_text(str(year))

    # ----------------------------------------------------------
    # C. GET CURRENT STATISTICS
    # ----------------------------------------------------------

    row = df.loc[
        df['year'] == year
    ].iloc[0]

    area = row['built_surface_km2']

    growth = row['growth_percent']

    # ----------------------------------------------------------
    # D. UPDATE GRAPH
    # ----------------------------------------------------------

    progress_line.set_data(
        all_years[:frame + 1],
        all_areas[:frame + 1]
    )

    current_point.set_data(
        [year],
        [area]
    )

    # ----------------------------------------------------------
    # E. UPDATE NUMERICAL STATISTICS
    # ----------------------------------------------------------

    area_text.set_text(
        f'BUILT-UP SURFACE: {area:,.2f} km²'
    )



    return (
        map_image,
        year_label,
        progress_line,
        current_point,
        area_text,
        growth_text
    )


# ==============================================================
# 14. CREATE ANIMATION
# ==============================================================

# Two frames per year

frame_sequence = [
    i
    for i in range(len(years))
    for _ in range(2)
]

ani = animation.FuncAnimation(
    fig,
    update,
    frames=frame_sequence,
    interval=1000,
    blit=False,
    repeat=False
)


# ==============================================================
# 15. EXPORT MP4
# ==============================================================

print('\nExporting animation...')
print('Please wait...')

writer = animation.FFMpegWriter(
    fps=1,
    bitrate=6000,
    metadata={
        'title': 'The Expanding City: Tashkent',
        'artist': 'Tumaris Ramanova'
    }
)

ani.save(
    output_path,
    writer=writer,
    dpi=120
)

plt.close(fig)

print('\n====================================')
print('ANIMATION EXPORTED SUCCESSFULLY!')
print('====================================')

print('Saved to:', output_path)


# ==============================================================
# 16. DISPLAY FINAL VIDEO
# ==============================================================

display(
    Video(
        output_path,
        embed=True,
        width=1100
    )
)

# ==============================================================
# END OF DAY 09
# ==============================================================


Both input files found!

RASTER INFORMATION
CRS: EPSG:32642
Resolution: (100.0, 100.0)
Dimensions: 798 800
Extent: BoundingBox(left=480300.0, bottom=4532000.0, right=560100.0, top=4612000.0)

Urban development extent calculated!

HISTORICAL STATISTICS
 year  system:index  built_surface_km2  footprint_km2                                   .geo  growth_percent  net_increase_km2
 1975             0         147.785582     432.905742 {"type":"MultiPoint","coordinates":[]}        0.000000          0.000000
 1980             1         165.353835     494.842970 {"type":"MultiPoint","coordinates":[]}       11.887663         17.568253
 1985             2         188.782417     592.149132 {"type":"MultiPoint","coordinates":[]}       27.740754         40.996835
 1990             3         216.462065     689.931990 {"type":"MultiPoint","coordinates":[]}       46.470354         68.676483
 1995             4         249.856097     794.143339 {"type":"MultiPoint","coordinates":[]}       69.066626     